In [0]:
from pyspark.sql.types import *

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False).lower() == "true"
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, CATALOG)

In [0]:
TARGET_TABLE_CONF = {
    "table": "pyspark_static_location_lookup",
    "schema": "silver"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
middle_earth_city_lookup = [
    # city, realm_name, realm_code, region, language_region
    ("Minas Tirith",      "Gondor",         "GO", "Gondor",    "Westron"),
    ("Osgiliath",         "Gondor",         "GO", "Gondor",    "Westron"),
    ("Pelargir",          "Gondor",         "GO", "Gondor",    "Westron"),
    ("Dol Amroth",        "Gondor",         "GO", "Gondor",    "Westron/Sindarin"),
    ("Edoras",            "Rohan",          "RO", "Rohan",     "Rohirric"),
    ("Helm's Deep",       "Rohan",          "RO", "Rohan",     "Rohirric"),
    ("Aldburg",           "Rohan",          "RO", "Rohan",     "Rohirric"),
    ("Dunharrow",         "Rohan",          "RO", "Rohan",     "Rohirric"),
    ("Isengard",          "Isengard",       "IS", "Rohan",     "Westron"),
    ("Hobbiton",          "The Shire",      "SH", "Eriador",   "Westron"),
    ("Bywater",           "The Shire",      "SH", "Eriador",   "Westron"),
    ("Michel Delving",    "The Shire",      "SH", "Eriador",   "Westron"),
    ("Tuckborough",       "The Shire",      "SH", "Eriador",   "Westron"),
    ("Bree",              "Bree-land",      "BR", "Eriador",   "Westron"),
    ("Combe",             "Bree-land",      "BR", "Eriador",   "Westron"),
    ("Archet",            "Bree-land",      "BR", "Eriador",   "Westron"),
    ("Rivendell",         "Rivendell",      "RV", "Eriador",   "Sindarin"),
    ("Grey Havens",       "Lindon",         "LI", "Eriador",   "Sindarin"),
    ("Fornost",           "Arnor",          "AR", "Eriador",   "Westron"),
    ("Annúminas",         "Arnor",          "AR", "Eriador",   "Westron"),
    ("Tharbad",           "Enedwaith",      "EN", "Enedwaith", "Westron"),
    ("Esgaroth",          "Dale",           "DA", "Rhovanion", "Westron"),
    ("Dale",              "Dale",           "DA", "Rhovanion", "Westron"),
    ("Erebor",            "Erebor",         "ER", "Rhovanion", "Khuzdul"),
    ("Caras Galadhon",    "Lothlórien",     "LO", "Rhovanion", "Sindarin"),
    ("Thranduil's Halls", "Woodland Realm", "WR", "Rhovanion", "Sindarin"),
]

city_lookup_schema = StructType([
    StructField("city",            StringType()),
    StructField("realm_name",      StringType()),
    StructField("realm_code",      StringType()),
    StructField("region",          StringType()),
    StructField("language_region", StringType()),
])

df_city_lookup = spark.createDataFrame(middle_earth_city_lookup, city_lookup_schema)

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"

    (df_city_lookup
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())